# 03A - Running Experiments with Azure Machine Learning SDK v2

This notebook is the updated SDK v2 version of the older Azure Machine Learning experiment lab.

The notebook has been upgraded from the retired SDK v1 experiment style to the current SDK v2 pattern.

The SDK v2 pattern uses Azure ML jobs plus MLflow tracking.

In Azure ML v2, experiment tracking is done with **MLflow**. Jobs submitted to Azure ML appear under **Jobs** in Azure Machine Learning studio.

## 1. Install required packages

Run this once if your notebook environment does not already have the SDK v2 packages installed.


In [ ]:
# Uncomment and run if needed
# %pip install azure-ai-ml azure-identity mlflow pandas matplotlib -U


## 2. Connect to your Azure ML workspace

This expects a `config.json` file in the current folder or parent folders. If your environment does not have one, create/download it from Azure ML studio.


In [ ]:
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from azure.ai.ml import MLClient

try:
    credential = DefaultAzureCredential()
    # Force a token check so we know whether the credential works
    credential.get_token("https://management.azure.com/.default")
except Exception:
    credential = InteractiveBrowserCredential()

ml_client = MLClient.from_config(credential=credential)

workspace = ml_client.workspaces.get(ml_client.workspace_name)
print(f"Connected to workspace: {workspace.name}")
print(f"Resource group: {ml_client.resource_group_name}")
print(f"Subscription: {ml_client.subscription_id}")


## 3. Load and inspect the diabetes dataset locally

The dataset file must exist at `data/diabetes.csv`.


In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("data/diabetes.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Missing data/diabetes.csv. Put the diabetes.csv file inside a folder named data."
    )

data = pd.read_csv(DATA_PATH)
print(data.shape)
data.head()


## 4. Run an inline experiment with MLflow

This replaces the old SDK v1 `experiment.start_logging()` and `run.log()` approach.


In [ ]:
import mlflow
import matplotlib.pyplot as plt

# Point MLflow to the Azure ML workspace tracking server
mlflow_tracking_uri = workspace.mlflow_tracking_uri
mlflow.set_tracking_uri(mlflow_tracking_uri)

experiment_name = "diabetes-experiment-v2"
mlflow.set_experiment(experiment_name)

with mlflow.start_run(run_name="inline-data-exploration") as run:
    row_count = len(data)
    mlflow.log_metric("observations", row_count)
    print(f"Analyzing {row_count} rows of data")

    # Plot and log diabetic vs non-diabetic counts
    diabetic_counts = data["Diabetic"].value_counts()

    fig = plt.figure(figsize=(6, 6))
    ax = fig.gca()
    diabetic_counts.plot.bar(ax=ax)
    ax.set_title("Patients with Diabetes")
    ax.set_xlabel("Diagnosis")
    ax.set_ylabel("Patients")
    plt.show()

    plot_path = "label_distribution.png"
    fig.savefig(plot_path, bbox_inches="tight")
    mlflow.log_artifact(plot_path)

    # Log pregnancy categories as a parameter because MLflow params are searchable text values
    pregnancies = sorted(data["Pregnancies"].dropna().unique().tolist())
    mlflow.log_param("pregnancy_categories", ",".join(map(str, pregnancies)))

    # Log summary statistics
    med_columns = [
        "PlasmaGlucose",
        "DiastolicBloodPressure",
        "TricepsThickness",
        "SerumInsulin",
        "BMI",
    ]

    summary_stats = data[med_columns].describe()
    for column in summary_stats.columns:
        for stat_name, stat_value in summary_stats[column].items():
            metric_name = f"{column}_{stat_name}"
            mlflow.log_metric(metric_name, float(stat_value))

    # Save and log a sample file
    sample_path = "sample.csv"
    data.sample(100, random_state=42).to_csv(sample_path, index=False)
    mlflow.log_artifact(sample_path, artifact_path="outputs")

    run_id = run.info.run_id
    print(f"MLflow run ID: {run_id}")


## 5. View inline experiment results

In SDK v2, metrics and artifacts are retrieved through MLflow.


In [ ]:
client = mlflow.tracking.MlflowClient()
finished_run = client.get_run(run_id)

print("Run ID:", finished_run.info.run_id)
print("Status:", finished_run.info.status)
print("Artifact URI:", finished_run.info.artifact_uri)

print("
Metrics:")
for key, value in finished_run.data.metrics.items():
    print(f"- {key}: {value}")

print("
Artifacts:")
for artifact in client.list_artifacts(run_id):
    print(f"- {artifact.path}")


## 6. Create an experiment script

This replaces the old `%%writefile diabetes_experiment.py` script that used `Run.get_context()`.

In SDK v2, the script logs metrics and files with MLflow.


In [ ]:
import os
from pathlib import Path
import shutil

experiment_folder = Path("diabetes-experiment-v2-files")
experiment_folder.mkdir(exist_ok=True)

shutil.copy(DATA_PATH, experiment_folder / "diabetes.csv")
print(f"Experiment folder ready: {experiment_folder}")


In [ ]:
%%writefile diabetes-experiment-v2-files/diabetes_experiment.py

import argparse
from pathlib import Path
import pandas as pd
import mlflow

parser = argparse.ArgumentParser()
parser.add_argument("--data_path", type=str, default="diabetes.csv")
args = parser.parse_args()

data_path = Path(args.data_path)
if data_path.is_dir():
    data_path = data_path / "diabetes.csv"

if not data_path.exists():
    raise FileNotFoundError(f"Could not find diabetes dataset at: {data_path}")

data = pd.read_csv(data_path)

with mlflow.start_run():
    row_count = len(data)
    mlflow.log_metric("observations", row_count)
    print(f"Analyzing {row_count} rows of data")

    diabetic_counts = data["Diabetic"].value_counts()
    for label, count in diabetic_counts.items():
        mlflow.log_metric(f"Label_{label}", int(count))

    sample_path = "sample.csv"
    data.sample(100, random_state=42).to_csv(sample_path, index=False)
    mlflow.log_artifact(sample_path, artifact_path="outputs")

    print("Experiment script completed.")


## 7. Submit the script as an Azure ML command job

Change `compute_name` to the name of your Azure ML compute cluster. Example names are often `cpu-cluster`, `cpu-cluster-001`, or whatever you created in Azure ML studio.

If you do not have compute yet, create one in Azure ML studio under **Compute > Compute clusters**.


In [ ]:
from azure.ai.ml import command, Input
from azure.ai.ml.entities import Environment
from azure.ai.ml.constants import AssetTypes

compute_name = "cpu-cluster"  # Change this to your actual Azure ML compute name

job_env = Environment(
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest",
    conda_file={
        "channels": ["conda-forge"],
        "dependencies": [
            "python=3.10",
            "pip",
            "pandas",
            "matplotlib",
            {"pip": ["mlflow", "azureml-mlflow"]},
        ],
    },
)

job = command(
    code=str(experiment_folder),
    command="python diabetes_experiment.py --data_path ${{inputs.diabetes_data}}",
    inputs={
        "diabetes_data": Input(
            type=AssetTypes.URI_FILE,
            path=str(experiment_folder / "diabetes.csv"),
        )
    },
    environment=job_env,
    compute=compute_name,
    experiment_name="diabetes-experiment-v2",
    display_name="diabetes-script-job-v2",
)

returned_job = ml_client.jobs.create_or_update(job)
print(f"Submitted job: {returned_job.name}")
print(f"Studio URL: {returned_job.studio_url}")


## 8. Wait for the job and review results

The job can take a few minutes because Azure ML may need to start compute and build the environment.


In [ ]:
ml_client.jobs.stream(returned_job.name)

completed_job = ml_client.jobs.get(returned_job.name)
print("Job status:", completed_job.status)
print("Studio URL:", completed_job.studio_url)


## 9. View experiment run history

In SDK v2, Azure ML jobs are listed through `ml_client.jobs`. For MLflow tracking history, use `MlflowClient`.


In [ ]:
# Azure ML job history for this experiment
for job_item in ml_client.jobs.list():
    if job_item.experiment_name == "diabetes-experiment-v2":
        print(job_item.name, "-", job_item.status)


## What changed from the old notebook?

| Old approach | New SDK v2 replacement |
|---|---|
| SDK v1 workspace connection | `MLClient.from_config(...)` |
| SDK v1 experiment object | `experiment_name` on jobs and `mlflow.set_experiment(...)` |
| Manual SDK run logging | `mlflow.start_run()` |
| SDK metric logging | `mlflow.log_metric(...)` or `mlflow.log_param(...)` |
| SDK image logging | Save image, then `mlflow.log_artifact(...)` |
| SDK file upload | `mlflow.log_artifact(...)` |
| Script run context object | MLflow tracking inside the job script |
| Old script-run configuration | `command(...)` job |
| Notebook run-details widget | Azure ML studio job URL + MLflow client |

The important change is this: **SDK v2 uses MLflow for experiment tracking.**